# Canada Vapes Sales Data Cleaning

## Goal

Clean and validate the Canada Vapes sales dataset so it can be used for sales analysis and customer segmentation. The notebook:

- standardizes the supplied column names;
- cleans dates, currency, quantities, customer information, and coupons;
- identifies missing values, questionable records, and outliers without automatically deleting genuine high-value orders;
- creates useful business measures using missing-value-safe calculations; and
- exports cleaned, analysis-ready, and quality-review CSV files.

## Setup

Run the notebook from top to bottom. If the location of `cvd5.csv` changes, update `DEFAULT_INPUT_FILE` in the file-location cell.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

### 1. Set the input and output locations

In [2]:
# Update this path if the source file is moved.
DEFAULT_INPUT_FILE = Path(
    r"C:\Users\tooko\Downloads\canada vape\cvd5.csv"
)

# The environment-variable option is useful for testing; it does not
# change normal Windows use.
INPUT_FILE = Path(
    os.environ.get(
        "CANADA_VAPES_INPUT_FILE",
        str(DEFAULT_INPUT_FILE),
    )
)

OUTPUT_FOLDER = INPUT_FILE.parent
CLEANED_FILE = OUTPUT_FOLDER / "cleaned_sales_data.csv"
ANALYSIS_FILE = OUTPUT_FOLDER / "sales_analysis_ready.csv"
ISSUES_FILE = OUTPUT_FOLDER / "sales_data_quality_issues.csv"

print("Input file:", INPUT_FILE)
print("Output folder:", OUTPUT_FOLDER)

Input file: C:\Users\tooko\Downloads\canada vape\cvd5.csv
Output folder: C:\Users\tooko\Downloads\canada vape


### 2. Load the dataset

In [3]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"File not found: {INPUT_FILE}\n"
        "Update DEFAULT_INPUT_FILE so it points to cvd5.csv."
    )

file_extension = INPUT_FILE.suffix.lower()

if file_extension == ".csv":
    df = pd.read_csv(INPUT_FILE, low_memory=False)
elif file_extension in [".xlsx", ".xls"]:
    df = pd.read_excel(INPUT_FILE)
else:
    raise ValueError("The input file must be CSV, XLSX, or XLS.")

original_row_count = len(df)

print("Original dataset size:", df.shape)
print("Original columns:")
print(df.columns.tolist())
display(df.head())

Original dataset size: (181641, 11)
Original columns:
['Date', 'Order #', 'N. Revenue (formatted)', 'Status', 'Customer', 'Customer type', 'Product(s)', 'Items sold', 'Coupon(s)', 'Net Sales', 'Attribution']


,Date,Order #,N. Revenue (formatted),Status,Customer,Customer type,Product(s),Items sold,Coupon(s),Net Sales,Attribution
0,2026-01-01 23:53,"785,500.00",$68.98,completed,Cody Gallant,new,2× Mixed Berries - CV NOVA 30K,2,NaN,59.98,Organic: Google
1,2026-01-01 23:51,"785,499.00",$194.96,completed,doug falconer,returning,"4× Red Ultra Salts - 10 mg/ml HIGH, 60ml",4,wlr-ls6-e6g,187.22,Direct
2,2026-01-01 23:48,"785,498.00",$68.96,completed,Anne-Marie Thivierge,returning,"1× Mint - CV NOVA 30K, 1× Grape Ice - CV NOVA 30K",2,NaN,59.98,Organic: Google
3,2026-01-01 23:44,"785,497.00",$112.99,completed,Carter Penny,new,1× XMax V3 Pro Dry Herb &amp; Wax Vaporizer - ...,1,NaN,99.99,Referral: Search.brave.com
4,2026-01-01 23:22,"785,496.00",$153.70,completed,Stephen Brenner,returning,"2× No Flavour - 9 mg/ml MEDIUM, 120ml",2,NaN,147.60,Organic: Google


## Steps

### 3. Standardize and verify column names

Special characters in headers such as `Order #`, `Product(s)`, and `Coupon(s)` are converted into consistent Python-friendly names. The aliases below match the actual Canada Vapes file.

In [4]:
df.columns = (
    df.columns
      .astype(str)
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

column_aliases = {
    "order": "order_id",
    "order_number": "order_id",
    "order_no": "order_id",
    "order_date": "date",
    "n_revenue_formatted": "revenue",
    "n_revenue": "revenue",
    "revenue_formatted": "revenue",
    "customer": "customer_id",
    "product_s": "products",
    "products_s": "products",
    "product": "products",
    "coupon_s": "coupon",
    "coupons_s": "coupon",
    "coupons": "coupon",
}

for original_name, new_name in column_aliases.items():
    if original_name in df.columns and new_name not in df.columns:
        df = df.rename(columns={original_name: new_name})

required_columns = [
    "date", "order_id", "revenue", "status", "customer_id",
    "customer_type", "products", "items_sold", "coupon",
    "net_sales", "attribution",
]

missing_columns = [
    column for column in required_columns if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

print("Standardized columns:")
print(df.columns.tolist())
print("All required columns are available.")

Standardized columns:
['date', 'order_id', 'revenue', 'status', 'customer_id', 'customer_type', 'products', 'items_sold', 'coupon', 'net_sales', 'attribution']
All required columns are available.


### 4. Replace missing-value symbols and remove exact duplicates

In [5]:
df = df.replace(r"^\s*$", np.nan, regex=True)

missing_symbols = [
    "N/A", "n/a", "NA", "na", "NULL", "null", "None", "none",
    "Missing", "missing", "-", "--",
]
df = df.replace(missing_symbols, np.nan)

duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

print("Exact duplicate rows removed:", duplicate_count)
print("Dataset size after duplicate removal:", df.shape)
print("\nMissing values after symbol replacement:")
display(df[required_columns].isna().sum().to_frame("missing_rows"))

Exact duplicate rows removed: 0
Dataset size after duplicate removal: (181641, 11)

Missing values after symbol replacement:


,missing_rows
date,0
order_id,3
revenue,3
status,0
customer_id,51656
customer_type,0
products,68224
items_sold,0
coupon,158162
net_sales,0


### 5. Clean order and customer identifiers

Exact duplicate rows are removed, but duplicate order numbers are retained because one order may contain more than one product row.

In [6]:
df["order_id"] = (
    df["order_id"]
      .astype("string")
      .str.strip()
      .str.replace(r"\.0$", "", regex=True)
      .replace(["", "nan", "None", "<NA>"], pd.NA)
)

df["customer_id"] = (
    df["customer_id"]
      .astype("string")
      .str.strip()
      .str.replace(r"\s+", " ", regex=True)
      .replace(["", "nan", "None", "<NA>"], pd.NA)
)

display(df[["order_id", "customer_id"]].head())

,order_id,customer_id
0,785500,Cody Gallant
1,785499,doug falconer
2,785498,Anne-Marie Thivierge
3,785497,Carter Penny
4,785496,Stephen Brenner


### 6. Clean the date and create time variables

In [7]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# If the source uses day/month/year, use this version instead:
# df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")

df["year"] = df["date"].dt.year.astype("Int64")
df["month_number"] = df["date"].dt.month.astype("Int64")
df["month_name"] = df["date"].dt.month_name()
df["quarter"] = df["date"].dt.to_period("Q").astype("string")
df["year_month"] = df["date"].dt.to_period("M").astype("string")
df["day_of_week"] = df["date"].dt.day_name()

print("Invalid or missing dates:", int(df["date"].isna().sum()))
display(
    df[["date", "year", "month_name", "quarter", "day_of_week"]].head()
)

Invalid or missing dates: 0


,date,year,month_name,quarter,day_of_week
0,2026-01-01 23:53:00,2026,January,2026Q1,Thursday
1,2026-01-01 23:51:00,2026,January,2026Q1,Thursday
2,2026-01-01 23:48:00,2026,January,2026Q1,Thursday
3,2026-01-01 23:44:00,2026,January,2026Q1,Thursday
4,2026-01-01 23:22:00,2026,January,2026Q1,Thursday


### 7. Clean currency and items sold

In [8]:
def clean_currency(column):
    """Convert formatted currency text into nullable numeric values."""
    text = (
        column.astype("string")
              .str.strip()
              .str.replace("−", "-", regex=False)
    )

    bracket_negative = text.str.match(r"^\(.*\)$", na=False)
    cleaned = text.str.replace(r"[^\d.\-()]", "", regex=True)
    cleaned = cleaned.str.replace(r"[()]", "", regex=True)
    cleaned = pd.to_numeric(cleaned, errors="coerce")
    cleaned.loc[bracket_negative] = -cleaned.loc[bracket_negative].abs()
    return cleaned


df["revenue"] = clean_currency(df["revenue"])
df["net_sales"] = clean_currency(df["net_sales"])

df["items_sold"] = (
    df["items_sold"]
      .astype("string")
      .str.replace(",", "", regex=False)
      .str.strip()
)
df["items_sold"] = pd.to_numeric(df["items_sold"], errors="coerce")

display(df[["revenue", "net_sales", "items_sold"]].describe())

,revenue,net_sales,items_sold
count,"180,498.00","181,641.00","181,641.00"
mean,126.31,100.72,2.97
std,"1,648.95",75.94,2.33
min,0.00,-559.90,-18.00
25%,67.75,58.64,2.00
50%,92.01,80.98,2.00
75%,142.56,126.06,4.00
max,"361,349.66","3,967.69",92.00


### 8. Standardize text, order status, customer type, and coupons

In [9]:
text_columns = ["status", "customer_type", "products", "attribution"]

for column in text_columns:
    df[column] = (
        df[column]
          .astype("string")
          .str.strip()
          .str.replace(r"\s+", " ", regex=True)
    )

df["status"] = df["status"].str.title()
df["customer_type"] = df["customer_type"].str.title()
df["attribution"] = df["attribution"].str.title()

status_mapping = {
    "Complete": "Completed", "Completed": "Completed",
    "Paid": "Completed", "Successful": "Completed",
    "Success": "Completed", "Cancel": "Cancelled",
    "Canceled": "Cancelled", "Cancelled": "Cancelled",
    "Refund": "Refunded", "Refunded": "Refunded",
    "Returned": "Refunded", "Return": "Refunded",
    "Pending": "Pending", "Processing": "Pending",
    "In Progress": "Pending",
}
df["status"] = df["status"].replace(status_mapping)

customer_type_mapping = {
    "New Customer": "New", "New": "New", "First Time": "New",
    "First-Time": "New", "First Purchase": "New",
    "Returning Customer": "Returning",
    "Return Customer": "Returning", "Returning": "Returning",
    "Existing": "Returning", "Existing Customer": "Returning",
    "Repeat": "Returning",
}
df["customer_type"] = df["customer_type"].replace(customer_type_mapping)

coupon_text = df["coupon"].astype("string").str.strip()
coupon_lower = coupon_text.str.lower()
no_coupon_values = [
    "", "no", "n", "false", "0", "none", "no coupon",
    "not used", "not applicable",
]

df["coupon_used"] = (
    coupon_text.notna() & ~coupon_lower.isin(no_coupon_values)
)
df["coupon"] = coupon_text.str.upper().mask(
    coupon_lower.isin(no_coupon_values), pd.NA
)

categorical_columns = [
    "status", "customer_type", "products", "attribution"
]
df[categorical_columns] = df[categorical_columns].fillna("Unknown")

print("Order status:")
display(df["status"].value_counts(dropna=False).to_frame("rows"))
print("Customer type:")
display(df["customer_type"].value_counts(dropna=False).to_frame("rows"))
print("Coupon use:")
display(df["coupon_used"].value_counts(dropna=False).to_frame("rows"))

Order status:


,rows
status,
Completed,176216
Reshipped,2235
Curbside-Complete,1393
Home-Delivery-Com,833
Refunded,817
Processing-1,86
Fraud,59
Processing-Reship,1
On-Hold-Custom,1


Customer type:


,rows
customer_type,
Returning,132165
New,49476


Coupon use:


,rows
coupon_used,
False,158162
True,23479


### 9. Create data-quality flags

In [10]:
df["missing_order_id_flag"] = df["order_id"].isna()
df["missing_customer_id_flag"] = df["customer_id"].isna()
df["missing_date_flag"] = df["date"].isna()
df["missing_revenue_flag"] = df["revenue"].isna()
df["missing_net_sales_flag"] = df["net_sales"].isna()
df["missing_items_sold_flag"] = df["items_sold"].isna()

df["negative_revenue_flag"] = df["revenue"].lt(0).fillna(False)
df["negative_net_sales_flag"] = df["net_sales"].lt(0).fillna(False)
df["negative_items_flag"] = df["items_sold"].lt(0).fillna(False)

df["non_integer_items_flag"] = (
    df["items_sold"].notna() & df["items_sold"].mod(1).ne(0)
).fillna(False)

df["net_sales_above_revenue_flag"] = (
    df["net_sales"].notna()
    & df["revenue"].notna()
    & df["net_sales"].gt(df["revenue"])
).fillna(False)

df["cancelled_with_sales_flag"] = (
    df["status"].eq("Cancelled") & df["net_sales"].gt(0)
).fillna(False)

print("Initial quality flags created.")

Initial quality flags created.


### 10. Flag outliers

The IQR method identifies unusual values but does not remove them. A large order may be a real, valuable transaction rather than an error.

In [11]:
def add_outlier_flag(data, column):
    valid_values = data[column].dropna()
    flag_name = f"{column}_outlier_flag"

    if valid_values.empty:
        data[flag_name] = False
        return data

    q1 = valid_values.quantile(0.25)
    q3 = valid_values.quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - (1.5 * iqr)
    upper_limit = q3 + (1.5 * iqr)

    data[flag_name] = (
        data[column].notna()
        & (
            data[column].lt(lower_limit)
            | data[column].gt(upper_limit)
        )
    ).fillna(False)

    print(
        f"{column}: lower={lower_limit:,.2f}, "
        f"upper={upper_limit:,.2f}, "
        f"outliers={int(data[flag_name].sum()):,}"
    )
    return data


for numeric_column in ["revenue", "net_sales", "items_sold"]:
    df = add_outlier_flag(df, numeric_column)

revenue: lower=-44.47, upper=254.78, outliers=9,841
net_sales: lower=-42.49, upper=227.19, outliers=10,697
items_sold: lower=-1.00, upper=7.00, outliers=7,158


### 11. Create business variables safely

These calculations use Pandas conditions instead of `numpy.where`. This avoids the `TypeError: boolean value of NA is ambiguous` error when revenue or items sold contains a missing value.

In [12]:
df["sales_deduction"] = df["revenue"] - df["net_sales"]

valid_revenue = df["revenue"].fillna(0).gt(0)
df["deduction_percentage"] = (
    df["sales_deduction"]
      .div(df["revenue"])
      .mul(100)
      .where(valid_revenue, np.nan)
)

valid_items = df["items_sold"].fillna(0).gt(0)
df["net_sales_per_item"] = (
    df["net_sales"]
      .div(df["items_sold"])
      .where(valid_items, np.nan)
)

display(
    df[[
        "revenue", "net_sales", "sales_deduction",
        "deduction_percentage", "items_sold", "net_sales_per_item",
    ]].head()
)

,revenue,net_sales,sales_deduction,deduction_percentage,items_sold,net_sales_per_item
0,68.98,59.98,9.00,13.05,2,29.99
1,194.96,187.22,7.74,3.97,4,46.80
2,68.96,59.98,8.98,13.02,2,29.99
3,112.99,99.99,13.00,11.51,1,99.99
4,153.70,147.60,6.10,3.97,2,73.80


### 12. Create the overall quality flag and sort the records

In [13]:
quality_flag_columns = [
    "missing_order_id_flag", "missing_customer_id_flag",
    "missing_date_flag", "missing_revenue_flag",
    "missing_net_sales_flag", "missing_items_sold_flag",
    "negative_revenue_flag", "negative_net_sales_flag",
    "negative_items_flag", "non_integer_items_flag",
    "net_sales_above_revenue_flag", "cancelled_with_sales_flag",
    "revenue_outlier_flag", "net_sales_outlier_flag",
    "items_sold_outlier_flag",
]

df["data_quality_issue"] = (
    df[quality_flag_columns].fillna(False).any(axis=1)
)

df = (
    df.sort_values(["date", "order_id"], na_position="last")
      .reset_index(drop=True)
)

print("Rows with at least one quality flag:", int(df["data_quality_issue"].sum()))

Rows with at least one quality flag: 64409


### 13. Create the analysis and quality-review datasets

In [14]:
analysis_df = (
    df.dropna(subset=["date", "order_id", "net_sales"])
      .copy()
      .reset_index(drop=True)
)

quality_issues_df = (
    df.loc[df["data_quality_issue"]]
      .copy()
      .reset_index(drop=True)
)

print(f"Cleaned rows: {len(df):,}")
print(f"Analysis-ready rows: {len(analysis_df):,}")
print(f"Quality-review rows: {len(quality_issues_df):,}")

Cleaned rows: 181,641
Analysis-ready rows: 181,638
Quality-review rows: 64,409


## Checks

### 14. Review the final cleaning results

In [15]:
cleaning_summary = pd.DataFrame(
    {
        "measure": [
            "Original rows",
            "Rows after exact duplicate removal",
            "Analysis-ready rows",
            "Quality-review rows",
            "Unique orders",
            "Unique customers",
        ],
        "value": [
            original_row_count,
            len(df),
            len(analysis_df),
            len(quality_issues_df),
            df["order_id"].nunique(dropna=True),
            df["customer_id"].nunique(dropna=True),
        ],
    }
)

display(cleaning_summary)

print("Quality-flag totals:")
display(
    df[quality_flag_columns]
      .sum()
      .sort_values(ascending=False)
      .to_frame("flagged_rows")
)

print("Financial summary:")
display(
    analysis_df[[
        "revenue", "net_sales", "items_sold",
        "sales_deduction", "net_sales_per_item",
    ]].describe()
)

,measure,value
0,Original rows,181641
1,Rows after exact duplicate removal,181641
2,Analysis-ready rows,181638
3,Quality-review rows,64409
4,Unique orders,180333
5,Unique customers,22723


Quality-flag totals:


,flagged_rows
missing_customer_id_flag,51656
net_sales_outlier_flag,10697
revenue_outlier_flag,9841
items_sold_outlier_flag,7158
negative_net_sales_flag,1188
missing_revenue_flag,1143
negative_items_flag,808
missing_order_id_flag,3
missing_date_flag,0
missing_net_sales_flag,0


Financial summary:


,revenue,net_sales,items_sold,sales_deduction,net_sales_per_item
count,"180,498.00","181,638.00","181,638.00","180,498.00","180,315.00"
mean,126.31,100.72,2.97,25.26,39.49
std,"1,648.95",75.94,2.33,"1,646.98",21.88
min,0.00,-559.90,-18.00,0.00,-25.48
25%,67.75,58.64,2.00,8.36,22.94
50%,92.01,80.98,2.00,11.99,33.33
75%,142.56,126.06,4.00,17.32,52.99
max,"361,349.66","3,967.69",92.00,"361,361.00",184.48


### 15. Save the cleaned files

In [16]:
df.to_csv(CLEANED_FILE, index=False)
analysis_df.to_csv(ANALYSIS_FILE, index=False)
quality_issues_df.to_csv(ISSUES_FILE, index=False)

print("Files saved successfully:")
print("1.", CLEANED_FILE)
print("2.", ANALYSIS_FILE)
print("3.", ISSUES_FILE)

Files saved successfully:
1. C:\Users\tooko\Downloads\canada vape\cleaned_sales_data.csv
2. C:\Users\tooko\Downloads\canada vape\sales_analysis_ready.csv
3. C:\Users\tooko\Downloads\canada vape\sales_data_quality_issues.csv


## Next Steps

- Use `sales_analysis_ready.csv` for revenue, product, coupon, attribution, and order-status analysis.
- Use `customer_id` to build customer-level RFM and behavioural segmentation features.
- Review `sales_data_quality_issues.csv` before deciding whether flagged refunds, negative quantities, or unusually large orders are genuine.
- Do not automatically delete IQR outliers; investigate them first.